In [92]:
import pandas as pd
import numpy as np
import glob
import os
from tqdm.auto import tqdm

files = glob.glob(
    "Data/extracted_data_extended/group*/experiment*/subject*.csv"
)

output_dir = "Data/extracted_data_aggregated"

os.makedirs(output_dir, exist_ok=True)

window = pd.Timedelta(milliseconds=500)

for file in tqdm(files, desc="Processing files"):

    print(f"\nProcessing: {file}")
    df = pd.read_csv(file)

    # split image and sensor rows
    images_df = df[df["image_path"].notna()].copy()
    sensors_df = df[df["image_path"].isna()].copy()

    # skip empty
    if len(images_df) == 0:
        continue

    # same time format
    # images_df["time"] = pd.to_datetime(images_df["time"], format="mixed")
    # sensors_df["time"] = pd.to_datetime(sensors_df["time"], format="mixed")

    images_df["time"] = pd.to_timedelta(images_df["time"].astype(str))
    sensors_df["time"] = pd.to_timedelta(sensors_df["time"].astype(str))
    
    images_df = images_df.sort_values("time")
    sensors_df = sensors_df.sort_values("time")

    # SENSOR COLUMNS
    sensor_columns = [
        col for col in sensors_df.columns
        if (
            "value" in col
            and sensors_df[col].notna().any()
        )
    ]

    # AGGREGATION
    aggregated_rows = []

    for _, img_row in tqdm(
        images_df.iterrows(),
        total=len(images_df),
        leave=False,
        desc="Aggregating"
    ):

        t = img_row["time"]

        nearby = sensors_df[
            (sensors_df["time"] >= t - window) &
            (sensors_df["time"] <= t + window)
        ]

        new_row = img_row.to_dict()

        # SENSOR AXIS MEAN / STD

        for col in sensor_columns:

            values = nearby[col].dropna()

            if len(values) == 0:
                new_row[f"{col}_mean"] = np.nan
                new_row[f"{col}_std"] = np.nan

            else:
                new_row[f"{col}_mean"] = values.mean()
                new_row[f"{col}_std"] = values.std()

        # compute magnitude for acceleration and gyroscope
        
        accel_cols = [
            "samsung_linear_acceleration_sensor value0",
            "samsung_linear_acceleration_sensor value1",
            "samsung_linear_acceleration_sensor value2"
        ]

        if all(col in nearby.columns for col in accel_cols):

            accel_valid = nearby[accel_cols].dropna()
            if len(accel_valid) > 0:

                accel_magnitude = np.sqrt(
                    accel_valid[accel_cols[0]]**2 +
                    accel_valid[accel_cols[1]]**2 +
                    accel_valid[accel_cols[2]]**2
                )

                new_row["accel_magnitude_mean"] = accel_magnitude.mean()
                new_row["accel_magnitude_std"] = accel_magnitude.std()

            else:

                new_row["accel_magnitude_mean"] = np.nan
                new_row["accel_magnitude_std"] = np.nan

        gyro_cols = [
            "lsm6dso_gyroscope value0",
            "lsm6dso_gyroscope value1",
            "lsm6dso_gyroscope value2"
        ]

        if all(col in nearby.columns for col in gyro_cols):

            gyro_valid = nearby[gyro_cols].dropna()
            if len(gyro_valid) > 0:

                gyro_magnitude = np.sqrt(
                    gyro_valid[gyro_cols[0]]**2 +
                    gyro_valid[gyro_cols[1]]**2 +
                    gyro_valid[gyro_cols[2]]**2
                )

                new_row["gyro_magnitude_mean"] = gyro_magnitude.mean()
                new_row["gyro_magnitude_std"] = gyro_magnitude.std()

            else:

                new_row["gyro_magnitude_mean"] = np.nan
                new_row["gyro_magnitude_std"] = np.nan

        aggregated_rows.append(new_row)

    
    final_df = pd.DataFrame(aggregated_rows)

    # drop the initial sensor columns (NaNs)
    raw_sensor_cols = [
        col for col in final_df.columns
        if (
            "value" in col
            and not col.endswith("_mean")
            and not col.endswith("_std")
        )
    ]
    
    final_df = final_df.drop(columns=raw_sensor_cols)

    relative_path = os.path.relpath(
        file,
        "Data/extracted_data_extended")

    output_path = os.path.join(
        output_dir,
        relative_path)

    output_path = output_path.replace(
        ".csv",
        ".parquet")

    os.makedirs(
        os.path.dirname(output_path),
        exist_ok=True)
    
    final_df.to_parquet(
        output_path,
        engine="pyarrow",
        index=False)

    print(f"[SAVED] {output_path}")
    
    # clean memory
    del df
    del images_df
    del sensors_df
    del final_df

Processing files:   0%|          | 0/2 [00:00<?, ?it/s]


Processing: Data/extracted_data_extended\group01\experiment01\subject_01.csv


Aggregating:   0%|          | 0/2531 [00:00<?, ?it/s]


Processing: Data/extracted_data_extended\group01\experiment01\subject_02.csv


Aggregating:   0%|          | 0/2506 [00:00<?, ?it/s]

In [85]:
import pandas as pd
import numpy as np
import glob

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

files = glob.glob("Data/extracted_data_aggregated/group*/experiment*/subject*.parquet")
df = pd.concat((pd.read_parquet(f) for f in files), ignore_index=True)
# df.to_parquet("Data/initial_data.parquet", engine="pyarrow", index=False)

In [71]:
# df = pd.read_parquet("Data/initial_data.parquet", engine="pyarrow")1
# df2 = pd.read_parquet("Data/initial_sensors_metadata.parquet", engine="pyarrow")

# tst = df.copy()
# tst = tst.drop(columns=['image_path', 'metadata'])

# tst = tst.sort_values(['group', 'experiment', 'subject', 'time'])
# df2 = df2.sort_values(['group', 'experiment', 'subject', 'time'])

# cols_to_add = [
#     'image_path',
#     'metadata',
#     'age',
#     'gender_name',
#     'race',
#     'race_asian',
#     'race_black',
#     'race_indian',
#     'race_latino_hispanic',
#     'race_middle_eastern',
#     'race_white'
# ]

# tst3 = pd.concat(
#     [tst, df2[cols_to_add].reset_index(drop=True)],
#     axis=1
# )


# # move image_path and metadata to 3rd and 4th column positions

# cols = tst3.columns.tolist()

# # remove the columns first
# cols.remove('image_path')
# cols.remove('metadata')

# # insert them at desired positions
# cols.insert(2, 'image_path')   # 3rd column (0-based index)
# cols.insert(3, 'metadata')     # 4th column

# tst3 = tst3[cols]

## Extract metadata (gender, age, race)

The following cell was executed in Snellius cluster where all the metadata folders for each subject are stored. As an output we receive the same df + 3 columns of age, gender and race for each row. It will output the file "initial_sensors_metadata.parquet" which we use in the subsequent cells.

In [ ]:
# df = pd.read_parquet("Data/initial_data.parquet", engine="pyarrow", index=False)

In [14]:
# Metadata extraction function
def extract_metadata(json_path):
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        face = data.get("person", {}).get("face", {})

        age = face.get("age", None)
        gender = face.get("gender", {}).get("gender_name", None)
        race = face.get("race", {}).get("dominant_race", None)

        # race probabilities
        prob_race = face.get("race", {}).get("probability_race", {})

        return (
            age,
            gender,
            race,
            prob_race.get("asian", None),
            prob_race.get("indian", None),
            prob_race.get("black", None),
            prob_race.get("white", None),
            prob_race.get("middle eastern", None),
            prob_race.get("latino hispanic", None),
        )

    except Exception:
        return (None,) * 9

# Parallel processing
def process_dataframe(df, n_jobs):
    paths = df["metadata"].tolist()
    results = []

    for i, p in enumerate(paths):
        results.append(extract_metadata(p))

        if i % 5000 == 0:
            print(f"Processed {i}/{len(paths)} rows")

    df[[
    "age",
    "gender_name",
    "race",
    "race_asian",
    "race_indian",
    "race_black",
    "race_white",
    "race_middle_eastern",
    "race_latino_hispanic"
    ]] = pd.DataFrame(results, index=df.index)
    return df

def main():
    start_time = time.time()
    parser = argparse.ArgumentParser()
    parser.add_argument("--input", required=True, help="Input dataframe path (parquet)")
    parser.add_argument("--output", required=True, help="Output dataframe path")
    parser.add_argument("--n_jobs", type=int, default=16)

    args = parser.parse_args()

    print("Loading dataframe...")
    df = pd.read_parquet(args.input)

    print(f"Processing {len(df)} rows with {args.n_jobs} workers...")

    df = process_dataframe(df, args.n_jobs)

    print("Saving output...")
    df.to_parquet(args.output)

    print("Done.")
    print(f"Elapsed: {time.time() - start_time:.2f}s")

# df.to_parquet("Data/initial_metadata.parquet", engine="pyarrow")

### Drop records with noise in image

In [83]:
df = pd.read_parquet("Data/initial_metadata.parquet", engine="pyarrow")

In [84]:
df.isna().sum()

group                        0
time                         0
image_path                   0
metadata                     0
subject                      0
                         ...  
race_black              146879
race_indian             146879
race_latino_hispanic    146879
race_middle_eastern     146879
race_white              146879
Length: 69, dtype: int64

In [ ]:
import tarfile
from tqdm import tqdm
import os

The objective of this process is to identify frames that could introduce noise in the pipeline, such as images where the face is non visible or covered. 

The following cells were executed in Snellius cluster where all the metadata folders for each subject are stored in tar fornat. As an output we receive the same df + 2 columns of valid (True - False), and the reason if False. It will output the file "initial_sensors_metadata_valid.parquet" which we use in the subsequent cells.

metadata folders are in format "metadata.tar" for each subject due to inode constraints for Snellius.

In [ ]:
def get_tar_and_filename(metadata_path):
    """
    Input:
        /.../subject_01/metadata/10_59_49_000776.json

    Output:
        (/.../subject_01/metadata.tar, 10_59_49_000776.json)
    """

    parts = metadata_path.split("/")

    try:
        meta_idx = parts.index("metadata")
    except ValueError:
        return None, None

    base = "/".join(parts[:meta_idx])  # up to subject_01
    filename = parts[-1]

    tar_path = f"{base}/metadata.tar"

    return tar_path, filename

def load_tar_index(tar_path):
    tar = tarfile.open(tar_path, "r")
    index = {}

    for m in tar.getmembers():
        if m.isfile() and m.name.endswith(".json"):
            fname = m.name.split("/")[-1]
            index[fname] = m

    return tar, index


def read_metadata_from_tar(tar, member):
    f = tar.extractfile(member)
    if f is None:
        return None
    return json.load(f)

In [ ]:
def is_valid_frame(metadata_path, metadata):

    if metadata is None:
        print(metadata_path, "rejected: metadata is None")
        return False, "metadata_none"

    person = metadata.get("person")
    # if person is None:
    #     print(metadata_path, "rejected: missing person")
    #     return False, "missing_person"

    face = person.get("face")
    if face is None:
        print(metadata_path, "rejected: missing face")
        return False, "missing_face"

    # --- Face bounding box ---
    bbox = face.get("bounding_box")
    
    if bbox is None:
        print(metadata_path, "rejected: missing bbox")
        return False, "missing_bbox"
    
    # CASE 1: dict format
    if isinstance(bbox, dict):
    
        x0 = bbox.get("x0")
        y0 = bbox.get("y0")
        x1 = bbox.get("x1")
        y1 = bbox.get("y1")
    
    # CASE 2: list format
    elif isinstance(bbox, list):
    
        if len(bbox) != 4:
            print(metadata_path, "rejected: malformed bbox list")
            return False, "malformed_bbox"
    
        x0, y0, x1, y1 = bbox
    
    # UNKNOWN FORMAT
    else:
        print(metadata_path, "rejected: unknown bbox format")
        return False, "unknown_bbox_format"
    
    # validate coords
    if None in (x0, y0, x1, y1):
        print(metadata_path, "rejected: incomplete bbox")
        return False, "incomplete_bbox"
    
    width = x1 - x0
    height = y1 - y0
    
    if width <= 0 or height <= 0:
        print(metadata_path, "rejected: invalid bbox dimensions")
        return False, "invalid_bbox"
        
    # --- Body pose sanity check ---
    # body = person.get("body")

    # if body:
    #     body_pose = body.get("body_pose") or []

    #     valid_landmarks = [
    #         lm for lm in body_pose
    #         if isinstance(lm, dict)
    #         and lm.get("visibility", 0) > 0.5
    #         and lm.get("presence", 0) > 0.5
    #     ]

    #     if len(valid_landmarks) < 5:
    #         print(metadata_path, "rejected: insufficient body landmarks")
    #         return False, "low_body_landmarks"

    return True, "valid"

In [ ]:
results = []
reasons = []

tar_cache = {}  # avoid reopening same tar

for _, row in tqdm(df.iterrows(), total=len(df)):

    metadata_path = row["metadata"]

    # Resolve tar path + filename
    tar_path, filename = get_tar_and_filename(metadata_path)

    if tar_path is None:
        results.append(None)
        reasons.append("invalid_tar_path")
        continue

    # Open tar (cached)
    try:

        if tar_path not in tar_cache:
            tar, index = load_tar_index(tar_path)
            tar_cache[tar_path] = (tar, index)

        tar, index = tar_cache[tar_path]

    except Exception as e:

        print(f"TAR OPEN ERROR: {metadata_path} -> {e}")

        results.append(None)
        reasons.append("tar_open_failure")
        continue

    # Find json inside tar
    member = index.get(filename)

    if member is None:

        print(f"MISSING JSON IN TAR: {metadata_path}")

        results.append(None)
        reasons.append("missing_json_in_tar")
        continue

    # Read metadata
    try:

        metadata = read_metadata_from_tar(tar, member)

    except Exception as e:

        print(f"METADATA READ ERROR: {metadata_path} -> {e}")

        results.append(None)
        reasons.append("metadata_read_failure")
        continue

    # Empty metadata
    if metadata is None or metadata == {}:

        print(f"EMPTY METADATA: {metadata_path}")

        results.append(None)
        reasons.append("empty_metadata")
        continue

    # Actual frame validation
    try:

        valid, reason = is_valid_frame(metadata_path, metadata)

        results.append(valid)
        reasons.append(reason)

    except Exception as e:

        print(f"VALIDATION ERROR: {metadata_path} -> {e}")

        results.append(None)
        reasons.append("validation_failure")


for tar, _ in tar_cache.values():
    tar.close()


df["valid"] = results
df["invalid_reason"] = reasons

In [ ]:
# df.to_parquet("Data/dipser_data_valid.parquet", engine="pyarrow", index=False)

### Data Cleaning

In [146]:
df = pd.read_parquet("Data/dipser_data_valid.parquet", engine="pyarrow")

In [147]:
df['invalid_reason'].value_counts()

invalid_reason
valid                    1059808
missing_face               46881
missing_bbox                2714
metadata_read_failure       2082
validation_failure             8
Name: count, dtype: int64

In [148]:
# the "None" values where with cases where metadata was empty. we set these rows to True as there is no evidence they are noise
# cases where invalid_reason in ['metadata_read_failure', 'validation_failure']
df["valid"] = df["valid"].astype("boolean").fillna(True)

In [149]:
df.shape

(1111493, 71)

In [150]:
# remove rows with noisy images
df = df[df['valid'] == True]

In [151]:
df.shape

(1061898, 71)

### Transform Metadata

After removing noisy images, with the remaining rows we can compute the fairness labels of gender, age and race

In [152]:
race_cols = [
    "race_asian",
    "race_indian",
    "race_black",
    "race_white",
    "race_middle_eastern",
    "race_latino_hispanic"
]

In [153]:
subject_probs = (
    df.groupby(["group", "subject"])[race_cols]
    .mean()
    .reset_index()
)

In [154]:
"""Deepface extracts per each timeframe the estimation for race, gender and age.
 For the column of gender, we will keep the most common label for the whole subject."""

# we will groupby group experiment and subject and we will keep the most dominant 
def get_mode(series):
    return series.dropna().mode().iloc[0] if not series.dropna().empty else None

In [155]:
def get_dominant_race(row):
    return row[race_cols].idxmax().replace("race_", "")

In [156]:
subject_probs["race"] = subject_probs.apply(get_dominant_race, axis=1)
subject_probs = subject_probs.drop(columns=race_cols)

In [157]:
subject_metadata = (
    df.groupby(["group", "subject"])
    .agg({
        "gender_name": get_mode,
        "age": "mean"
    })
    .reset_index())

# merge with race
subject_metadata = subject_metadata.merge(
    subject_probs,
    on=["group", "subject"],
    how="left"
)

In [158]:
df = df.drop(columns=['race', 'gender_name', 'age']) # will be replaced with the new values
df = df.merge(
    subject_metadata,
    on=["group", "subject"],
    how="left")

In [159]:
df['age'] = round(df['age'])

In [160]:
df = df.drop(columns=race_cols)

### Remove subjects with missing sensors

In [161]:
df['samsung_hr_none_wakeup_sensor value0_mean'].isna().sum()

np.int64(155464)

In [162]:
df['subject_experiment_id'] = df['group'] + "_" + df['experiment'] + '_' + df['subject'] 

In [163]:
missing_ratio = (
    df.groupby('subject_experiment_id')['samsung_hr_none_wakeup_sensor value0_mean']
    .apply(lambda x: x.isna().mean())
)
missing_ratio.sort_values(ascending=False).head()

subject_experiment_id
group02_experiment08_subject_01    1.0
group02_experiment08_subject_10    1.0
group02_experiment09_subject_01    1.0
group01_experiment05_subject_14    1.0
group01_experiment05_subject_01    1.0
Name: samsung_hr_none_wakeup_sensor value0_mean, dtype: float64

In [164]:
len(set(df['subject_experiment_id']))

483

In [165]:
MISSING_SENSORS_THRESHOLD = 0.3

In [166]:
valid_subjects = missing_ratio[missing_ratio < MISSING_SENSORS_THRESHOLD].index
df = df[df["subject_experiment_id"].isin(valid_subjects)]

In [167]:
len(set(df['subject_experiment_id']))

408

In [168]:
df = df.drop(columns='subject_experiment_id')

In [169]:
df['samsung_hr_none_wakeup_sensor value0_mean'].isna().sum()

np.int64(10310)

In [170]:
df.shape

(899997, 65)

In [171]:
df = df.dropna(subset='samsung_hr_none_wakeup_sensor value0_mean')
df = df.dropna(subset='samsung_rotation_vector value0_mean') # few records

In [172]:
df.shape

(889687, 65)

### Set "Ground Truth" Attention label

In [173]:
attention_cols = [
    col for col in df.columns
    if 'attentionfilled' in col and 'self' not in col]

# use averaging to serve as a ground truth label
df['attention'] = df[attention_cols].mean(axis=1)

### Unique identifier for each subject

In [174]:
df['subject_id'] = df['group'] + "_" + df['subject'] 

In [175]:
df = df.rename(columns={'samsung_hr_none_wakeup_sensor value0_mean': 'heart_rate', 'samsung_hr_none_wakeup_sensor value0_std': 'heart_rate_std', 'gender_name': 'gender'})

In [179]:
# remaining nans only for cases where one of the labellers was not part of the evaluation of the subject
df.isna().sum().head(60)

group                                                  0
time                                                   0
image_path                                             0
metadata                                               0
subject                                                0
experiment                                             0
self_labeling emotion                             885766
self_labeling attention                           886101
labeler_02 attention                              887602
labeler_04 attention                              888157
labeler_02 emotion                                883650
labeler_01 attention                              871709
labeler_03 emotion                                881659
labeler_03 attention                              881030
labeler_01 emotion                                886169
labeler_04 emotion                                888151
self_labeling emotionfilled                         2009
self_labeling attentionfilled  

### Export to Parquet

In [180]:
df.to_parquet("Data/dipser_cleaned_dataset.parquet", engine="pyarrow", index=False)

### Add Sensor Data

The following cells were executed in Snellius cluster where all the sensors_data folders for each subject are stored. As an output we receive the same df + columns of sensors data. It will output the file "initial_sensors_data.parquet".

In [304]:
# df = pd.read_parquet("Data/initial_data.parquet", engine="pyarrow")

In [305]:
# import glob
# import json
# import pandas as pd
# import numpy as np
# from datetime import datetime
# from loguru import logger

In [306]:
# def build_sensor_intervals(sensor_folder):
#     sensor_files = glob.glob(sensor_folder)

#     intervals = []
#     data_cache = {}

#     for f in sensor_files:
#         with open(f, "r") as fp:
#             data = json.load(fp)

#         times = []

#         for sensor in data["data"].values():
#             for entry in sensor:                
#                 times.append(datetime.strptime(entry["timestamp"], "%H:%M:%S:%f"))

#         if not times:
#             continue

#         start = min(times)
#         end = max(times)

#         intervals.append((f, start, end))
#         data_cache[f] = data

#     # sort by start time
#     intervals.sort(key=lambda x: x[1])

#     return intervals, data_cache

In [10]:
# # def find_sensor_interval(target_time, intervals, idx):
# #     """
# #     idx = pointer (so we don't scan from start every time)
# #     """
# #     n = len(intervals)

# #     # move forward while target is beyond current interval
# #     while idx < n - 1 and target_time > intervals[idx][2]:
# #         idx += 1

# #     f, start, end = intervals[idx]

# #     if start <= target_time <= end:
# #         return f, idx

# #     return None, idx

# def find_sensor_interval(target_time, intervals, idx, max_diff_sec=0.5):
#     n = len(intervals)

#     # Move pointer forward
#     while idx < n - 1 and target_time > intervals[idx][2]:
#         idx += 1

#     candidates = []

#     # current
#     f, start, end = intervals[idx]
#     dist_current = min(
#         abs((target_time - start).total_seconds()),
#         abs((target_time - end).total_seconds())
#     )
#     candidates.append((dist_current, idx))

#     # previous
#     if idx > 0:
#         f_prev, s_prev, e_prev = intervals[idx - 1]
#         dist_prev = min(
#             abs((target_time - s_prev).total_seconds()),
#             abs((target_time - e_prev).total_seconds())
#         )
#         candidates.append((dist_prev, idx - 1))

#     # next
#     if idx < n - 1:
#         f_next, s_next, e_next = intervals[idx + 1]
#         dist_next = min(
#             abs((target_time - s_next).total_seconds()),
#             abs((target_time - e_next).total_seconds())
#         )
#         candidates.append((dist_next, idx + 1))

#     # pick closest
#     best_dist, best_idx = min(candidates, key=lambda x: x[0])

#     # if the frame does not correspond to any of the intervals, assign it to the closest one 
#     # as long as the difference is not higher than threshold (1 second)
#     if best_dist > max_diff_sec:
#         return None, idx

#     f_best = intervals[best_idx][0]

#     return f_best, best_idx

In [307]:
# # overall strength of movement, independent of direction
# def compute_magnitude(x, y, z):
#     return np.sqrt(x**2 + y**2 + z**2)

# def aggregate_sensor(sensor_data, sensor_name):
#     features = {}

#     if len(sensor_data) == 0:
#         return features

#     values = {}
#     for key in sensor_data[0].keys():
#         if key.startswith("value"):
#             values[key] = np.array([d[key] for d in sensor_data])

#     if sensor_name in ["samsung_linear_acceleration_sensor", "lsm6dso_gyroscope"]:
#         x, y, z = values["value0"], values["value1"], values["value2"]
#         mag = compute_magnitude(x, y, z)

#         features.update({
#             f"{sensor_name}_mean_x": x.mean(),
#             f"{sensor_name}_std_x": x.std(),
#             f"{sensor_name}_mean_y": y.mean(),
#             f"{sensor_name}_std_y": y.std(),
#             f"{sensor_name}_mean_z": z.mean(),
#             f"{sensor_name}_std_z": z.std(),
#             f"{sensor_name}_mean_mag": mag.mean(),
#             f"{sensor_name}_std_mag": mag.std(),
#         })

#     elif sensor_name == "samsung_rotation_vector":
#         for k, v in values.items():
#             features[f"{sensor_name}_mean_{k}"] = v.mean()
#             features[f"{sensor_name}_std_{k}"] = v.std()

#     elif sensor_name == "opt3007_light":
#         v = values["value0"]
#         features[f"{sensor_name}_mean"] = v.mean()
#         features[f"{sensor_name}_std"] = v.std()
        
#     elif sensor_name == "samsung_hr_none_wakeup_sensor":
#         v = values["value0"]
#         features[f"{sensor_name}_value"] = v.mean()
#     return features

# def aggregate_all_sensors(data):
#     all_features = {}
#     for sensor_name, sensor_data in data["data"].items():
#         feats = aggregate_sensor(sensor_data, sensor_name)
#         all_features.update(feats)

#     return all_features

In [12]:
# def add_sensors_to_df(df, sensor_folder):
#     intervals, data_cache = build_sensor_intervals(sensor_folder)

#     sensor_rows = []
#     idx = 0  # pointer

#     for i, row in df.iterrows():
#         target_time = datetime.strptime(row["time"], "%H:%M:%S.%f")

#         sensor_file, idx = find_sensor_interval(target_time, intervals, idx)

#         if sensor_file is None:
#             sensor_rows.append({})
#             logger.warning(f"No close sensor match for time {target_time}")
#             continue

#         sensor_json = data_cache[sensor_file]
#         features = aggregate_all_sensors(sensor_json)
#         sensor_rows.append(features)

#     sensor_df = pd.DataFrame(sensor_rows)

#     return pd.concat([df.reset_index(drop=True), sensor_df], axis=1)

In [310]:
# def aggregate_sensor(sensor_data, sensor_name):
#     features = {}

#     if len(sensor_data) == 0:
#         return features

#     values = {}
#     for key in sensor_data[0].keys():
#         if key.startswith("value"):
#             values[key] = np.array([d[key] for d in sensor_data])

#     if sensor_name in ["samsung_linear_acceleration_sensor", "lsm6dso_gyroscope"]:
#         x, y, z = values["value0"], values["value1"], values["value2"]
#         mag = compute_magnitude(x, y, z)

#         features.update({
#             f"{sensor_name}_mean_x": x.mean(),
#             f"{sensor_name}_std_x": x.std(),
#             f"{sensor_name}_mean_y": y.mean(),
#             f"{sensor_name}_std_y": y.std(),
#             f"{sensor_name}_mean_z": z.mean(),
#             f"{sensor_name}_std_z": z.std(),
#             f"{sensor_name}_mean_mag": mag.mean(),
#             f"{sensor_name}_std_mag": mag.std(),
#         })

#     elif sensor_name == "samsung_rotation_vector":
#         for k, v in values.items():
#             features[f"{sensor_name}_mean_{k}"] = v.mean()
#             features[f"{sensor_name}_std_{k}"] = v.std()

#     elif sensor_name == "opt3007_light":
#         v = values["value0"]
#         features[f"{sensor_name}_mean"] = v.mean()
#         features[f"{sensor_name}_std"] = v.std()
        
#     elif sensor_name == "samsung_hr_none_wakeup_sensor":

#         v = values["value0"]    
#         # remove invalid HR values
#         v = v[(v > 30) & (v < 220)]
#         if len(v) > 0:
#             features[f"{sensor_name}_mean"] = v.mean()
#             features[f"{sensor_name}_std"] = v.std()
            
#     else:
#         features[f"{sensor_name}_mean"] = np.nan
#         features[f"{sensor_name}_std"] = np.nan
        
#     return features

# def aggregate_all_sensors(data):
#     all_features = {}
#     for sensor_name, sensor_data in data["data"].items():
#         feats = aggregate_sensor(sensor_data, sensor_name)
#         all_features.update(feats)

#     return all_features

In [311]:
# def add_sensors_to_df(df, sensor_folder):

#     intervals, data_cache = build_sensor_intervals(sensor_folder)

#     sensor_rows = []

#     idx = 0

#     for _, row in df.iterrows():

#         target_time = datetime.strptime(
#             row["time"],
#             "%H:%M:%S.%f"
#         )

#         sensor_file, idx = find_sensor_file(
#             target_time,
#             intervals,
#             idx
#         )

#         if sensor_file is None:

#             sensor_rows.append({})

#             logger.warning(
#                 f"No sensor file for time {target_time}"
#             )

#             continue

#         sensor_json = data_cache[sensor_file]

#         features = {}

#         for sensor_name, sensor_data in sensor_json["data"].items():

#             local_window = get_local_sensor_window(
#                 sensor_data,
#                 target_time,
#                 radius_sec=0.5
#             )

#             sensor_features = aggregate_sensor(
#                 local_window,
#                 sensor_name
#             )

#             features.update(sensor_features)

#         sensor_rows.append(features)

#     sensor_df = pd.DataFrame(sensor_rows)

#     return pd.concat(
#         [df.reset_index(drop=True), sensor_df],
#         axis=1
#     )

In [308]:
# def find_sensor_file(target_time, intervals, idx):

#     n = len(intervals)

#     while idx < n - 1 and target_time > intervals[idx][2]:
#         idx += 1

#     f, start, end = intervals[idx]

#     if start <= target_time <= end:
#         return f, idx

#     return None, idx

In [309]:
# def get_local_sensor_window(sensor_data, target_time, radius_sec=0.5):

#     selected = []

#     for entry in sensor_data:

#         t = datetime.strptime(
#             entry["timestamp"],
#             "%H:%M:%S:%f"
#         )

#         diff = abs((t - target_time).total_seconds())

#         if diff <= radius_sec:
#             selected.append(entry)

#     return selected

In [312]:
# # example (stored locally)
# sub10ex2 = df[(df['subject'] == 'subject_10') & (df['experiment'] == 'experiment02') & (df['group'] == 'group01')]
# sensor_folder = r"Data\DIPSER\group01\experiment02\subject_10\watch_sensors\*.json"
# sub10ex2_enriched = add_sensors_to_df(sub10ex2, sensor_folder)
# sub10ex2_enriched

2026-05-07 19:37:10.026 | WARNING  | __main__:add_sensors_to_df:26 - No sensor file for time 1900-01-01 10:59:50.384486
2026-05-07 19:37:10.059 | WARNING  | __main__:add_sensors_to_df:26 - No sensor file for time 1900-01-01 10:59:52.390421
2026-05-07 19:37:10.070 | WARNING  | __main__:add_sensors_to_df:26 - No sensor file for time 1900-01-01 10:59:53.392499
2026-05-07 19:37:10.201 | WARNING  | __main__:add_sensors_to_df:26 - No sensor file for time 1900-01-01 11:00:00.391111
2026-05-07 19:37:10.312 | WARNING  | __main__:add_sensors_to_df:26 - No sensor file for time 1900-01-01 11:00:06.433615
2026-05-07 19:37:10.334 | WARNING  | __main__:add_sensors_to_df:26 - No sensor file for time 1900-01-01 11:00:07.423867
2026-05-07 19:37:10.458 | WARNING  | __main__:add_sensors_to_df:26 - No sensor file for time 1900-01-01 11:00:13.444136
2026-05-07 19:37:10.521 | WARNING  | __main__:add_sensors_to_df:26 - No sensor file for time 1900-01-01 11:00:16.430030
2026-05-07 19:37:10.547 | WARNING  | __m

,group,time,subject,experiment,image_path,metadata,labeler_02 emotion,labeler_02 attention,labeler_01 attention,labeler_04 attention,...,samsung_linear_acceleration_sensor_mean_x,samsung_linear_acceleration_sensor_std_x,samsung_linear_acceleration_sensor_mean_y,samsung_linear_acceleration_sensor_std_y,samsung_linear_acceleration_sensor_mean_z,samsung_linear_acceleration_sensor_std_z,samsung_linear_acceleration_sensor_mean_mag,samsung_linear_acceleration_sensor_std_mag,samsung_hr_none_wakeup_sensor_mean,samsung_hr_none_wakeup_sensor_std
0,group01,10:59:49.008949,subject_10,experiment02,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,7.0,4.0,5.0,3.0,...,-0.043346,0.257764,0.377950,0.941372,0.264364,0.705908,1.010721,0.802489,73.0,0.0
1,group01,10:59:49.117509,subject_10,experiment02,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,NaN,NaN,NaN,NaN,...,-0.050821,0.274529,0.475393,0.969750,0.278749,0.754611,1.108358,0.814275,73.0,0.0
2,group01,10:59:49.251676,subject_10,experiment02,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,NaN,NaN,NaN,NaN,...,-0.060365,0.301143,0.614996,1.013276,0.260105,0.834982,1.246506,0.842823,73.0,0.0
3,group01,10:59:49.344585,subject_10,experiment02,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,NaN,NaN,NaN,NaN,...,-0.054146,0.325490,0.680091,1.081182,0.221095,0.898181,1.361259,0.861940,73.0,0.0
4,group01,10:59:49.436001,subject_10,experiment02,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,NaN,NaN,NaN,NaN,...,0.076530,0.234296,0.310154,0.667115,-0.232994,0.307177,0.690299,0.523566,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2191,group01,11:04:48.553386,subject_10,experiment02,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,NaN,NaN,NaN,NaN,...,0.042298,0.072961,0.016106,0.178214,-0.038162,0.247722,0.285947,0.142088,NaN,NaN
2192,group01,11:04:48.686729,subject_10,experiment02,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,NaN,NaN,NaN,NaN,...,0.043150,0.073470,0.017657,0.179370,-0.036403,0.260601,0.297608,0.142872,73.0,0.0
2193,group01,11:04:48.777590,subject_10,experiment02,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,NaN,NaN,NaN,NaN,...,0.042175,0.076156,0.008963,0.176290,-0.038000,0.273628,0.307073,0.144081,73.0,0.0
2194,group01,11:04:48.872595,subject_10,experiment02,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,NaN,NaN,NaN,NaN,...,0.047121,0.074383,-0.004407,0.164509,-0.020021,0.277688,0.300308,0.148869,73.0,0.0


In [313]:
# df2 = pd.read_parquet("Data/dipser_transformed_data.parquet", engine="pyarrow")  # approx match
# sub10ex2df2 = df2[(df2['subject'] == 'subject_10') & (df2['experiment'] == 'experiment02') & (df2['group'] == 'group01')]

In [315]:
# sub10ex2df2['samsung_hr_none_wakeup_sensor_value'].mean()

np.float64(75.3620218579235)

In [317]:
# sub10ex2_enriched['samsung_hr_none_wakeup_sensor_mean'].mean()

np.float64(76.33969465648855)